In [1]:
from pyspark import SparkContext

sc = SparkContext("local[*]", "DistributedPageRank")

links = [
    ("A", ["B", "C"]),
    ("B", ["C"]),
    ("C", ["A", "D"]),
    ("D", ["A"])
]

linksRDD = sc.parallelize(links)

N = linksRDD.count()
d = 0.85

ranks = linksRDD.map(lambda x: (x[0], 1.0 / N))

for i in range(20):

    contributions = linksRDD.join(ranks).flatMap(
        lambda x: [
            (dest, x[1][1] / len(x[1][0]))
            for dest in x[1][0]
        ]
    )

    newRanks = contributions.reduceByKey(
        lambda x, y: x + y
    ).mapValues(
        lambda rank: (1 - d) / N + d * rank
    )

    oldRanks = ranks
    ranks = newRanks

    difference = (
        oldRanks.join(ranks)
        .map(lambda x: abs(x[1][0] - x[1][1]))
        .sum()
    )



Iteration: 1
Difference: 0.42499999999999993
Iteration: 2
Difference: 0.18062499999999998
Iteration: 3
Difference: 0.07676562499999984
Iteration: 4
Difference: 0.03262539062499986
Iteration: 5
Difference: 0.013865791015625017
Iteration: 6
Difference: 0.005892961181640732
Iteration: 7
Difference: 0.0025045085021973557
Iteration: 8
Difference: 0.0010644161134338859
Iteration: 9
Difference: 0.0004523768482093127
Iteration: 10
Difference: 0.0001922601604888996
Iteration: 11
Difference: 8.17105682076602e-05
Iteration: 12
Difference: 3.472699148820979e-05
Iteration: 13
Difference: 1.4758971382644592e-05
Iteration: 14
Difference: 6.272562837639217e-06
Iteration: 15
Difference: 2.6658392058842573e-06
Iteration: 16
Difference: 1.132981662377297e-06
Iteration: 17
Difference: 4.815172065519846e-07

Final PageRank:
A 0.3245614394113706
C 0.3245614394113706
B 0.1754385605886293
D 0.1754385605886293
